In [ ]:
!pip install --quiet --no-cache-dir --no-deps xformers==0.0.27
!pip install --quiet --no-cache-dir --no-deps trl==0.8.6
!pip install --quiet --no-cache-dir --no-deps peft accelerate bitsandbytes
!pip install --quiet --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet datasets pandas pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 325.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 244.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 435.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 209.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 403.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 243.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 256.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 456.9 

In [ ]:
import os
!mkdir -p /content/code_fixer_v1
!unzip -q /content/code_fixer_v1-20260331T121910Z-1-001.zip -d /content/code_fixer_v1
!ls /content/code_fixer_v1

replace /content/code_fixer_v1/code_fixer_v1/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: code_fixer_v1


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_PATH = "/content/code_fixer_v1/code_fixer_v1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=1024,
    load_in_4bit=True,
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Model loaded safely")


==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.18 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


Model loaded safely


In [ ]:
from datasets import load_dataset

dataset = load_dataset("eitanturok/humaneval-fix-starcoder", split="test")

# keep it smaller for reliability
dataset = dataset.select(range(min(25, len(dataset))))

print("Loaded samples:", len(dataset))
print("Columns:", dataset.column_names)
print(dataset[0])


README.md:   0%|          | 0.00/572 [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/101k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded samples: 25
Columns: ['task_id', 'prompt', 'entry_point', 'canonical_solution', 'test', 'test_inputs', 'test_outputs', 'language']
{'task_id': 'HumanEval/1', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = elem - elem2\n                if distance < threshold:\n                    return True\n\n    return False\n\nFix bugs in has_close_elements.\n\nfrom typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:', 'entry_point': 'separate_paren_groups', 'canonical_solution': "    result = []\n    current_string = []\n    current_depth = 0\n\n    for c in paren_string:\n        if c == '(':\n            current_depth += 1\n            current_string.append(c)\n        elif c == ')':\n            current_depth -= 1\n            current_s

In [ ]:
import re
import ast
from collections import Counter

def build_eval_prompt(row):
    buggy_code = row["prompt"]
    parts = [
        {
            "role": "user",
            "content": (
                "Fix all bugs in this Python code.\n"
                "Return the complete corrected Python function or program.\n"
                "Do not return only a code fragment.\n"
                "Do not explain.\n"
                "Return code only.\n\n"
                f"{buggy_code}"
            )
        }
    ]
    return tokenizer.apply_chat_template(
        parts,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)

    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

def is_valid_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except:
        return False

def tokenize_for_metrics(text: str):
    return re.findall(r"\w+|[^\s\w]", text)

def precision_recall_f1(prediction: str, reference: str):
    pred_tokens = tokenize_for_metrics(prediction)
    ref_tokens = tokenize_for_metrics(reference)

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)

    common = pred_counter & ref_counter
    tp = sum(common.values())
    pred_total = sum(pred_counter.values())
    ref_total = sum(ref_counter.values())

    precision = tp / pred_total if pred_total else 0.0
    recall = tp / ref_total if ref_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return precision, recall, f1

def run_tests(prediction_code, test_code, entry_point):
    namespace = {}
    try:
        exec(prediction_code, namespace)
        exec(test_code, namespace)
        namespace["check"](namespace[entry_point])
        return True, ""
    except Exception as e:
        return False, str(e)


In [ ]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

results = []
save_path = "/content/eval_results_partial.csv"

for i, row in enumerate(tqdm(dataset)):
    prompt = build_eval_prompt(row)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False,
            temperature=0.1,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    prediction = clean_output(decoded)

    reference = row["canonical_solution"]
    syntax_valid = is_valid_python(prediction)
    exact_match = prediction.strip() == reference.strip()
    precision, recall, f1 = precision_recall_f1(prediction, reference)
    test_pass, test_error = run_tests(prediction, row["test"], row["entry_point"])

    results.append({
        "task_id": row["task_id"],
        "prediction": prediction,
        "reference": reference,
        "syntax_valid": syntax_valid,
        "exact_match": exact_match,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "test_pass": test_pass,
        "test_error": test_error,
    })

    pd.DataFrame(results).to_csv(save_path, index=False)

print(f"Saved results to {save_path}")
df = pd.DataFrame(results)
df.head()


  0%|          | 0/25 [00:00<?, ?it/s]Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

Saved results to /content/eval_results_partial.csv


,task_id,prediction,reference,syntax_valid,exact_match,precision,recall,f1,test_pass,test_error
0,HumanEval/1,"for idx, elem in enumerate(numbers):\n for ...",result = []\n current_string = []\n ...,True,False,0.418605,0.233766,0.300000,False,"'return' outside function (<string>, line 6)"
1,HumanEval/0,result = []\ncurrent_string = []\ncurrent_dept...,"for idx, elem in enumerate(numbers):\n ...",True,False,0.240506,0.441860,0.311475,False,"'return' outside function (<string>, line 18)"
2,HumanEval/2,return int(number),return number % 1.0\n,True,False,0.400000,0.333333,0.363636,False,"'return' outside function (<string>, line 1)"
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...,True,False,0.538462,1.000000,0.700000,True,
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...,True,False,0.625000,1.000000,0.769231,True,


In [ ]:
exact_match_accuracy = df["exact_match"].mean() * 100
syntax_valid_rate = df["syntax_valid"].mean() * 100
test_pass_rate = df["test_pass"].mean() * 100
avg_precision = df["precision"].mean() * 100
avg_recall = df["recall"].mean() * 100
avg_f1 = df["f1"].mean() * 100

print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")
print(f"Syntax Valid Rate: {syntax_valid_rate:.2f}%")
print(f"Test Pass Rate: {test_pass_rate:.2f}%")
print(f"Average Token Precision: {avg_precision:.2f}%")
print(f"Average Token Recall: {avg_recall:.2f}%")
print(f"Average Token F1: {avg_f1:.2f}%")


Exact Match Accuracy: 8.00%
Syntax Valid Rate: 88.00%
Test Pass Rate: 40.00%
Average Token Precision: 71.72%
Average Token Recall: 85.78%
Average Token F1: 76.35%


In [ ]:
print("Successful examples:")
display(df[df["test_pass"] == True][["task_id", "prediction"]].head(10))

print("\nFailed examples:")
display(df[df["test_pass"] == False][["task_id", "prediction", "test_error"]].head(10))


Successful examples:


,task_id,prediction
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...
5,HumanEval/5,from typing import List\n\n\ndef intersperse(n...
9,HumanEval/9,"from typing import List, Tuple\n\n\ndef rollin..."
11,HumanEval/11,"def string_xor(a: str, b: str) -> str:\n re..."
12,HumanEval/12,"from typing import List, Optional\n\n\ndef lon..."
13,HumanEval/13,"def greatest_common_divisor(a: int, b: int) ->..."
14,HumanEval/14,from typing import List\n\n\ndef all_prefixes(...
18,HumanEval/18,"def how_many_times(string: str, substring: str..."
21,HumanEval/21,from typing import List\n\n\ndef rescale_to_un...



Failed examples:


,task_id,prediction,test_error
0,HumanEval/1,"for idx, elem in enumerate(numbers):\n for ...","'return' outside function (<string>, line 6)"
1,HumanEval/0,result = []\ncurrent_string = []\ncurrent_dept...,"'return' outside function (<string>, line 18)"
2,HumanEval/2,return int(number),"'return' outside function (<string>, line 1)"
6,HumanEval/6,def parse_nested_parens(paren_string: str) -> ...,name 'List' is not defined
7,HumanEval/7,return [x for x in strings if substring in x],"'return' outside function (<string>, line 1)"
8,HumanEval/8,sum_value = 0\nprod_value = 1\n\nfor n in numb...,"'return' outside function (<string>, line 7)"
10,HumanEval/10,if not string:\n return ''\n\n begin...,unindent does not match any outer indentation ...
15,HumanEval/15,return ' '.join(str(x) for x in range(n)),"'return' outside function (<string>, line 1)"
16,HumanEval/16,return len(set(string)),"'return' outside function (<string>, line 1)"
17,HumanEval/17,"note_map = {'o': 3, 'o|': 2, '.|': 1}\n ret...","unexpected indent (<string>, line 2)"


In [ ]:
import re
import ast
from collections import Counter

def build_eval_prompt(row):
    buggy_code = row["prompt"]
    parts = [
        {
            "role": "user",
            "content": (
                "Fix all bugs in this Python code.\n"
                "Return the complete corrected Python function or program.\n"
                "Do not return only a code fragment.\n"
                "Preserve the function signature and required entry point.\n"
                "Do not explain.\n"
                "Return code only.\n\n"
                f"{buggy_code}"
            )
        }
    ]
    return tokenizer.apply_chat_template(
        parts,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_output(text: str) -> str:
    if not text:
        return ""
    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)
    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()
    return text.strip()

def is_valid_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except:
        return False

def tokenize_for_metrics(text: str):
    return re.findall(r"\w+|[^\s\w]", text)

def precision_recall_f1(prediction: str, reference: str):
    pred_tokens = tokenize_for_metrics(prediction)
    ref_tokens = tokenize_for_metrics(reference)

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    common = pred_counter & ref_counter

    tp = sum(common.values())
    pred_total = sum(pred_counter.values())
    ref_total = sum(ref_counter.values())

    precision = tp / pred_total if pred_total else 0.0
    recall = tp / ref_total if ref_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1

def run_tests(prediction_code, test_code, entry_point):
    namespace = {}
    try:
        exec(prediction_code, namespace)
        exec(test_code, namespace)
        namespace["check"](namespace[entry_point])
        return True, ""
    except Exception as e:
        return False, str(e)

def candidate_score(code, entry_point):
    score = 0
    if is_valid_python(code):
        score += 3
    if f"def {entry_point}" in code:
        score += 5
    if "return" in code:
        score += 1
    return score


In [ ]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

results = []

for row in tqdm(dataset):
    prompt = build_eval_prompt(row)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    best_prediction = ""
    best_score = -1

    generation_settings = [
        {"do_sample": False, "temperature": 0.1},
        {"do_sample": True, "temperature": 0.2},
        {"do_sample": True, "temperature": 0.4},
    ]

    for settings in generation_settings:
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=160,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                **settings
            )

        input_len = inputs["input_ids"].shape[1]
        generated_tokens = outputs[0][input_len:]
        decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        prediction = clean_output(decoded)

        score = candidate_score(prediction, row["entry_point"])
        if score > best_score:
            best_score = score
            best_prediction = prediction

    reference = row["canonical_solution"]
    syntax_valid = is_valid_python(best_prediction)
    exact_match = best_prediction.strip() == reference.strip()
    precision, recall, f1 = precision_recall_f1(best_prediction, reference)
    test_pass, test_error = run_tests(best_prediction, row["test"], row["entry_point"])

    results.append({
        "task_id": row["task_id"],
        "prediction": best_prediction,
        "reference": reference,
        "syntax_valid": syntax_valid,
        "exact_match": exact_match,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "test_pass": test_pass,
        "test_error": test_error,
    })

df = pd.DataFrame(results)
df.head()


  0%|          | 0/25 [00:00<?, ?it/s]Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

,task_id,prediction,reference,syntax_valid,exact_match,precision,recall,f1,test_pass,test_error
0,HumanEval/1,"for idx, elem in enumerate(numbers):\n for ...",result = []\n current_string = []\n ...,True,False,0.418605,0.233766,0.300000,False,"'return' outside function (<string>, line 6)"
1,HumanEval/0,result = []\ncurrent_string = []\ncurrent_dept...,"for idx, elem in enumerate(numbers):\n ...",True,False,0.240506,0.441860,0.311475,False,"'return' outside function (<string>, line 18)"
2,HumanEval/2,return int(number),return number % 1.0\n,True,False,0.400000,0.333333,0.363636,False,"'return' outside function (<string>, line 1)"
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...,True,False,0.538462,1.000000,0.700000,True,
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...,True,False,0.625000,1.000000,0.769231,True,


In [ ]:
exact_match_accuracy = df["exact_match"].mean() * 100
syntax_valid_rate = df["syntax_valid"].mean() * 100
test_pass_rate = df["test_pass"].mean() * 100
avg_precision = df["precision"].mean() * 100
avg_recall = df["recall"].mean() * 100
avg_f1 = df["f1"].mean() * 100

print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")
print(f"Syntax Valid Rate: {syntax_valid_rate:.2f}%")
print(f"Test Pass Rate: {test_pass_rate:.2f}%")
print(f"Average Token Precision: {avg_precision:.2f}%")
print(f"Average Token Recall: {avg_recall:.2f}%")
print(f"Average Token F1: {avg_f1:.2f}%")


Exact Match Accuracy: 8.00%
Syntax Valid Rate: 92.00%
Test Pass Rate: 40.00%
Average Token Precision: 71.27%
Average Token Recall: 86.52%
Average Token F1: 76.41%


In [ ]:
!pip install --quiet flask flask-cors nest-asyncio

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import torch
import gc
import re
import threading
import traceback

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

model_lock = threading.Lock()

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)

    fenced = re.search(r"```(?:\w+)?\n?([\s\S]*?)```", text)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

def looks_like_code(text: str) -> bool:
    if not text:
        return False

    lowered = text.strip().lower()

    # Reject obvious non-code outputs
    if lowered.startswith("[") and lowered.endswith("]") and "\n" not in lowered:
        return False
    if lowered.startswith("{") and lowered.endswith("}") and "\n" not in lowered:
        return False
    if re.fullmatch(r"[-+]?(\d+(\.\d+)?)(,\s*[-+]?(\d+(\.\d+)?))*", lowered):
        return False

    code_markers = [
        "def ", "class ", "if ", "for ", "while ", "try:", "except",
        "print(", "return ", "import ", "from ", "=", ":"
    ]
    return any(marker in text for marker in code_markers)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/fix", methods=["POST", "OPTIONS"])
def fix_code():
    if request.method == "OPTIONS":
        return jsonify({"status": "ok"}), 200

    try:
        data = request.get_json(silent=True) or {}
        user_code = (data.get("code") or "").strip()

        if not user_code:
            return jsonify({"error": "No code provided"}), 400

        prompt = (
            "<|im_start|>system\n"
            "You are a Python code repair model.\n"
            "Rewrite buggy Python code into corrected Python code.\n"
            "Return only the full corrected Python code.\n"
            "Do not explain anything.\n"
            "Do not summarize.\n"
            "Do not return program output.\n"
            "Do not return lists, answers, or comments unless they are part of the corrected code.\n"
            "Return valid Python code only.\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"{user_code}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
        )

        with model_lock:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=False
            )

            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=192,
                    do_sample=False,
                    temperature=0.1,
                    use_cache=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_len = inputs["input_ids"].shape[1]
            generated_tokens = outputs[0][input_len:]
            decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            fixed_code = clean_output(decoded)

        if not fixed_code:
            return jsonify({"error": "Model returned empty output"}), 500

        if not looks_like_code(fixed_code):
            return jsonify({"error": f"Model returned non-code output: {fixed_code}"}), 500

        return jsonify({"fixed_code": fixed_code}), 200

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


In [ ]:
import socket
import threading
import nest_asyncio

nest_asyncio.apply()

def find_free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = find_free_port()

def run_app():
    app.run(host="0.0.0.0", port=PORT, debug=False, threaded=False, use_reloader=False)

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

print(f"Flask server started on port {PORT}")


Flask server started on port 39051


In [ ]:
import os
import subprocess
import time
import requests

NGROK_AUTH_TOKEN = "3BiWDV0h8Uxh9U3w1PuTFKDLE5d_2VrYYqoyeuboBZSqS5smm"

!wget -q -nc https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin

os.system(f"ngrok config add-authtoken {NGROK_AUTH_TOKEN}")

ngrok_process = subprocess.Popen(
    ["ngrok", "http", str(PORT), "--log=stdout"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(25):
    time.sleep(2)
    try:
        res = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=5)
        data = res.json()
        for tunnel in data.get("tunnels", []):
            url = tunnel.get("public_url", "")
            if url.startswith("https://"):
                public_url = url
                break
        if public_url:
            break
    except:
        pass

if not public_url:
    raise Exception("Ngrok did not start correctly.")

print("Fix endpoint:", public_url + "/fix")
print("Health endpoint:", public_url + "/health")
print(requests.get(public_url + "/health", timeout=20, headers={"ngrok-skip-browser-warning": "true"}).text)


INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 10:29:30] "GET /health HTTP/1.1" 200 -


Fix endpoint: https://exportable-juli-nonobligatorily.ngrok-free.dev/fix
Health endpoint: https://exportable-juli-nonobligatorily.ngrok-free.dev/health
{"status":"ok"}



In [ ]:
def test_clean_output():
    sample_1 = "<|im_start|>assistant\n```python\ndef add(a,b):\n    return a+b\n```\n<|im_end|>"
    cleaned_1 = clean_output(sample_1)
    assert "```" not in cleaned_1
    assert "<|im_start|>" not in cleaned_1
    assert "def add(a,b):" in cleaned_1

    sample_2 = "assistant: def hello():\n    print('hi')"
    cleaned_2 = clean_output(sample_2)
    assert cleaned_2.startswith("def hello():")

    print("clean_output() unit tests passed")

def looks_like_code_local(text: str) -> bool:
    if not text:
        return False
    markers = ["def ", "class ", "if ", "for ", "while ", "try:", "except", "print(", "return ", "import ", "="]
    return any(marker in text for marker in markers)

def test_code_validation():
    assert looks_like_code_local("def add(a, b):\n    return a + b") == True
    assert looks_like_code_local("[20.0, None, 10.0]") == False
    assert looks_like_code_local("The answer is 15") == False
    print("code validation unit tests passed")

test_clean_output()
test_code_validation()
print("All unit tests passed")


clean_output() unit tests passed
code validation unit tests passed
All unit tests passed


In [ ]:
import requests

base_url = public_url

health_response = requests.get(base_url + "/health", timeout=20, headers={"ngrok-skip-browser-warning": "true"})
print("Health status code:", health_response.status_code)
print("Health response:", health_response.text)

assert health_response.status_code == 200
assert "ok" in health_response.text.lower()

payload = {
    "code": "def add(a, b)\n    return a + b\n\nprint(add(2, 3))"
}

fix_response = requests.post(
    base_url + "/fix",
    json=payload,
    timeout=120,
    headers={"ngrok-skip-browser-warning": "true"}
)

print("Fix status code:", fix_response.status_code)
print("Fix response:", fix_response.text)

assert fix_response.status_code == 200
assert "fixed_code" in fix_response.json()

print("Integration testing passed")


INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 10:49:00] "GET /health HTTP/1.1" 200 -


Health status code: 200
Health response: {"status":"ok"}



Both `max_new_tokens` (=192) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Fix status code: 200
Fix response: {"fixed_code":"def add(a, b):\n    return a + b\n\nprint(add(2, 3))"}

Integration testing passed


In [ ]:
import requests
import ast
system_test_cases = [
    {
        "name": "Syntax Error Fix",
        "input": "def greet(name)\n    print('Hello ' + name)\n\ngreet('Vinay')"
    },
    {
        "name": "Loop Syntax Fix",
        "input": "for i in range(3)\n    print(i)"
    },
    {
        "name": "Function Definition Fix",
        "input": "def square(x)\n    return x * x\n\nprint(square(4))"
    }
]
results = []
for case in system_test_cases:
    response = requests.post(
        public_url + "/fix",
        json={"code": case["input"]},
        timeout=120,
        headers={"ngrok-skip-browser-warning": "true"}
    )
    status = response.status_code
    data = response.json() if response.headers.get("content-type", "").startswith("application/json") else {}
    fixed_code = data.get("fixed_code", "")
    syntax_valid = False
    if fixed_code:
        try:
            ast.parse(fixed_code)
            syntax_valid = True
        except:
            syntax_valid = False
    results.append({
        "test_name": case["name"],
        "status_code": status,
        "syntax_valid": syntax_valid,
        "output_preview": fixed_code[:200]
    })

for r in results:
    print(r)

print("System testing complete")


Both `max_new_tokens` (=192) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 10:50:27] "POST /fix HTTP/1.1" 200 -
Both `max_new_tokens` (=192) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 10:50:31] "POST /fix HTTP/1.1" 200 -
Both `max_new_tokens` (=192) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 10:50:36] "POST /fix HTTP/1.1" 200 -


{'test_name': 'Syntax Error Fix', 'status_code': 200, 'syntax_valid': True, 'output_preview': "def greet(name):\n    print('Hello ' + name)\n\ngreet('Vinay')"}
{'test_name': 'Loop Syntax Fix', 'status_code': 200, 'syntax_valid': True, 'output_preview': 'for i in range(3):\n    print(i)'}
{'test_name': 'Function Definition Fix', 'status_code': 200, 'syntax_valid': True, 'output_preview': 'def square(x):\n    return x * x\n\nprint(square(4))'}
System testing complete


In [ ]:
from datasets import load_dataset

dataset = load_dataset("eitanturok/humaneval-fix-starcoder", split="test")
dataset = dataset.select(range(min(25, len(dataset))))

print("Loaded samples:", len(dataset))
print("Columns:", dataset.column_names)
print(dataset[0])


Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded samples: 25
Columns: ['task_id', 'prompt', 'entry_point', 'canonical_solution', 'test', 'test_inputs', 'test_outputs', 'language']
{'task_id': 'HumanEval/1', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = elem - elem2\n                if distance < threshold:\n                    return True\n\n    return False\n\nFix bugs in has_close_elements.\n\nfrom typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:', 'entry_point': 'separate_paren_groups', 'canonical_solution': "    result = []\n    current_string = []\n    current_depth = 0\n\n    for c in paren_string:\n        if c == '(':\n            current_depth += 1\n            current_string.append(c)\n        elif c == ')':\n            current_depth -= 1\n            current_s

In [ ]:
import re
import ast
from collections import Counter

def build_eval_prompt(row):
    buggy_code = row["prompt"]
    parts = [
        {
            "role": "user",
            "content": (
                "Fix all bugs in this Python code.\n"
                "Return the complete corrected Python function or program.\n"
                "Do not return only a code fragment.\n"
                "Do not explain.\n"
                "Return code only.\n\n"
                f"{buggy_code}"
            )
        }
    ]
    return tokenizer.apply_chat_template(
        parts,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)

    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

def is_valid_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except:
        return False

def tokenize_for_metrics(text: str):
    return re.findall(r"\w+|[^\s\w]", text)

def precision_recall_f1(prediction: str, reference: str):
    pred_tokens = tokenize_for_metrics(prediction)
    ref_tokens = tokenize_for_metrics(reference)

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    common = pred_counter & ref_counter

    tp = sum(common.values())
    pred_total = sum(pred_counter.values())
    ref_total = sum(ref_counter.values())

    precision = tp / pred_total if pred_total else 0.0
    recall = tp / ref_total if ref_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return precision, recall, f1

def run_tests(prediction_code, test_code, entry_point):
    namespace = {}
    try:
        exec(prediction_code, namespace)
        exec(test_code, namespace)
        namespace["check"](namespace[entry_point])
        return True, ""
    except Exception as e:
        return False, str(e)


In [ ]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

results = []
save_path = "/content/eval_results_partial.csv"

for row in tqdm(dataset):
    prompt = build_eval_prompt(row)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False,
            temperature=0.1,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    prediction = clean_output(decoded)

    reference = row["canonical_solution"]
    syntax_valid = is_valid_python(prediction)
    exact_match = prediction.strip() == reference.strip()
    precision, recall, f1 = precision_recall_f1(prediction, reference)
    test_pass, test_error = run_tests(prediction, row["test"], row["entry_point"])

    results.append({
        "task_id": row["task_id"],
        "prediction": prediction,
        "reference": reference,
        "syntax_valid": syntax_valid,
        "exact_match": exact_match,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "test_pass": test_pass,
        "test_error": test_error,
    })

    pd.DataFrame(results).to_csv(save_path, index=False)

print(f"Saved results to {save_path}")
df = pd.DataFrame(results)
df.head()

  0%|          | 0/25 [00:00<?, ?it/s]Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

Saved results to /content/eval_results_partial.csv


,task_id,prediction,reference,syntax_valid,exact_match,precision,recall,f1,test_pass,test_error
0,HumanEval/1,"for idx, elem in enumerate(numbers):\n for ...",result = []\n current_string = []\n ...,True,False,0.418605,0.233766,0.300000,False,"'return' outside function (<string>, line 6)"
1,HumanEval/0,result = []\ncurrent_string = []\ncurrent_dept...,"for idx, elem in enumerate(numbers):\n ...",True,False,0.240506,0.441860,0.311475,False,"'return' outside function (<string>, line 18)"
2,HumanEval/2,return int(number),return number % 1.0\n,True,False,0.400000,0.333333,0.363636,False,"'return' outside function (<string>, line 1)"
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...,True,False,0.538462,1.000000,0.700000,True,
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...,True,False,0.625000,1.000000,0.769231,True,


In [ ]:
exact_match_accuracy = df["exact_match"].mean() * 100
syntax_valid_rate = df["syntax_valid"].mean() * 100
test_pass_rate = df["test_pass"].mean() * 100
avg_precision = df["precision"].mean() * 100
avg_recall = df["recall"].mean() * 100
avg_f1 = df["f1"].mean() * 100

print(f"Exact Match Accuracy: {exact_match_accuracy:.2f}%")
print(f"Syntax Valid Rate: {syntax_valid_rate:.2f}%")
print(f"Test Pass Rate: {test_pass_rate:.2f}%")
print(f"Average Token Precision: {avg_precision:.2f}%")
print(f"Average Token Recall: {avg_recall:.2f}%")
print(f"Average Token F1: {avg_f1:.2f}%")

Exact Match Accuracy: 8.00%
Syntax Valid Rate: 88.00%
Test Pass Rate: 40.00%
Average Token Precision: 71.72%
Average Token Recall: 85.78%
Average Token F1: 76.35%


In [ ]:
from datasets import load_dataset

dataset = load_dataset("eitanturok/humaneval-fix-starcoder", split="test")
dataset = dataset.select(range(min(25, len(dataset))))

print("Loaded samples:", len(dataset))
print("Columns:", dataset.column_names)
print(dataset[0])


Loaded samples: 25
Columns: ['task_id', 'prompt', 'entry_point', 'canonical_solution', 'test', 'test_inputs', 'test_outputs', 'language']
{'task_id': 'HumanEval/1', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = elem - elem2\n                if distance < threshold:\n                    return True\n\n    return False\n\nFix bugs in has_close_elements.\n\nfrom typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:', 'entry_point': 'separate_paren_groups', 'canonical_solution': "    result = []\n    current_string = []\n    current_depth = 0\n\n    for c in paren_string:\n        if c == '(':\n            current_depth += 1\n            current_string.append(c)\n        elif c == ')':\n            current_depth -= 1\n            current_s

In [ ]:
import re
import ast
from collections import Counter
from unsloth import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
)

def build_eval_prompt(row):
    buggy_code = row["prompt"]
    parts = [
        {
            "role": "user",
            "content": (
                "Fix all bugs in this Python code.\n"
                "Return the complete corrected Python function or program.\n"
                "Do not return only a code fragment.\n"
                "Do not explain.\n"
                "Return code only.\n\n"
                f"{buggy_code}"
            )
        }
    ]
    return tokenizer.apply_chat_template(
        parts,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)

    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

def is_valid_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except:
        return False

def get_ast_tree(code: str):
    try:
        tree = ast.parse(code)
        return ast.unparse(tree)
    except:
        return None

def tokenize_for_metrics(text: str):
    return re.findall(r"\w+|[^\s\w]", text)

def precision_recall_f1(prediction: str, reference: str):
    pred_tokens = tokenize_for_metrics(prediction)
    ref_tokens = tokenize_for_metrics(reference)

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    common = pred_counter & ref_counter

    tp = sum(common.values())
    pred_total = sum(pred_counter.values())
    ref_total = sum(ref_counter.values())

    precision = tp / pred_total if pred_total else 0.0
    recall = tp / ref_total if ref_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return precision, recall, f1

def run_tests(prediction_code, test_code, entry_point):
    namespace = {}
    try:
        exec(prediction_code, namespace)
        exec(test_code, namespace)
        namespace["check"](namespace[entry_point])
        return True, ""
    except Exception as e:
        return False, str(e)


In [ ]:
import torch
import gc
import pandas as pd
from tqdm import tqdm

results = []
save_path = "/content/eval_results_with_structural_match.csv"

for row in tqdm(dataset):
    prompt = build_eval_prompt(row)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False,
            temperature=0.1,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    prediction = clean_output(decoded)

    reference = row["canonical_solution"]

    syntax_valid = is_valid_python(prediction)
    exact_match = prediction.strip() == reference.strip()

    norm_pred = get_ast_tree(prediction)
    norm_ref = get_ast_tree(reference)
    logical_match = (norm_pred == norm_ref) and (norm_pred is not None)

    precision, recall, f1 = precision_recall_f1(prediction, reference)
    test_pass, test_error = run_tests(prediction, row["test"], row["entry_point"])

    results.append({
        "task_id": row["task_id"],
        "prediction": prediction,
        "reference": reference,
        "syntax_valid": syntax_valid,
        "exact_match": exact_match,
        "logical_match": logical_match,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "test_pass": test_pass,
        "test_error": test_error,
    })

    pd.DataFrame(results).to_csv(save_path, index=False)

print(f"Saved results to {save_path}")
df = pd.DataFrame(results)
df.head()


  0%|          | 0/25 [00:00<?, ?it/s]Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

Saved results to /content/eval_results_with_structural_match.csv


,task_id,prediction,reference,syntax_valid,exact_match,logical_match,precision,recall,f1,test_pass,test_error
0,HumanEval/1,"for idx, elem in enumerate(numbers):\n for ...",result = []\n current_string = []\n ...,True,False,False,0.418605,0.233766,0.300000,False,"'return' outside function (<string>, line 6)"
1,HumanEval/0,result = []\ncurrent_string = []\ncurrent_dept...,"for idx, elem in enumerate(numbers):\n ...",True,False,False,0.240506,0.441860,0.311475,False,"'return' outside function (<string>, line 18)"
2,HumanEval/2,return int(number),return number % 1.0\n,True,False,False,0.400000,0.333333,0.363636,False,"'return' outside function (<string>, line 1)"
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...,True,False,False,0.538462,1.000000,0.700000,True,
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...,True,False,False,0.625000,1.000000,0.769231,True,


In [ ]:
strict_exact_match_accuracy = df["exact_match"].mean() * 100
logical_match_rate = df["logical_match"].mean() * 100
syntax_valid_rate = df["syntax_valid"].mean() * 100
test_pass_rate = df["test_pass"].mean() * 100
avg_precision = df["precision"].mean() * 100
avg_recall = df["recall"].mean() * 100
avg_f1 = df["f1"].mean() * 100

print(f"Strict Exact Match Accuracy: {strict_exact_match_accuracy:.2f}%")
print(f"Normalized Structural Match Rate: {logical_match_rate:.2f}%")
print(f"Syntax Valid Rate: {syntax_valid_rate:.2f}%")
print(f"Test Pass Rate: {test_pass_rate:.2f}%")
print(f"Average Token Precision: {avg_precision:.2f}%")
print(f"Average Token Recall: {avg_recall:.2f}%")
print(f"Average Token F1: {avg_f1:.2f}%")


Strict Exact Match Accuracy: 8.00%
Normalized Structural Match Rate: 0.00%
Syntax Valid Rate: 88.00%
Test Pass Rate: 40.00%
Average Token Precision: 71.72%
Average Token Recall: 85.78%
Average Token F1: 76.35%


In [ ]:
manual_cases = [
    {
        "id": 1,
        "bug_type": "Syntax Error",
        "code": "def greet(name)\n    print('Hello ' + name)\n\ngreet('Vinay')"
    },
    {
        "id": 2,
        "bug_type": "Loop Syntax",
        "code": "for i in range(3)\n    print(i)"
    },
    {
        "id": 3,
        "bug_type": "Function Definition",
        "code": "def square(x)\n    return x * x\n\nprint(square(4))"
    },
    {
        "id": 4,
        "bug_type": "Mutable Default Argument",
        "code": "def add_item(item, my_list=[]):\n    my_list.append(item)\n    return my_list\n\nprint(add_item(1))\nprint(add_item(2))"
    },
    {
        "id": 5,
        "bug_type": "Recursion/Base Case",
        "code": "def factorial(n):\n    return n * factorial(n-1)\n\nprint(factorial(5))"
    },
    {
        "id": 6,
        "bug_type": "Class Definition",
        "code": "class Student\n    def __init__(self, name):\n        self.name = name\n\ns = Student('Vinay')\nprint(s.name)"
    },
    {
        "id": 7,
        "bug_type": "Conditional Syntax",
        "code": "x = 10\nif x > 5\n    print('Greater')"
    },
    {
        "id": 8,
        "bug_type": "Indentation Error",
        "code": "def add(a, b):\nreturn a + b\n\nprint(add(2, 3))"
    },
    {
        "id": 9,
        "bug_type": "String + Integer Concatenation",
        "code": "age = 20\nprint('Age is ' + age)"
    },
    {
        "id": 10,
        "bug_type": "Object Method Syntax",
        "code": "class Car:\n    def __init__(self, brand):\n        self.brand = brand\n    def show(self)\n        print(self.brand)\n\nc = Car('Toyota')\nc.show()"
    }
]

print("Manual evaluation cases loaded:", len(manual_cases))


Manual evaluation cases loaded: 10


In [ ]:
import torch
import gc
import pandas as pd
import re

def build_manual_prompt(code):
    parts = [
        {
            "role": "user",
            "content": (
                "Fix all bugs in this Python code.\n"
                "Return the complete corrected Python code only.\n"
                "Do not explain.\n\n"
                f"{code}"
            )
        }
    ]
    return tokenizer.apply_chat_template(
        parts,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_output(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE)

    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

manual_results = []

for case in manual_cases:
    prompt = build_manual_prompt(case["code"])

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=160,
            do_sample=False,
            temperature=0.1,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    prediction = clean_output(decoded)

    manual_results.append({
        "id": case["id"],
        "bug_type": case["bug_type"],
        "input_code": case["code"],
        "predicted_fix": prediction,
        "manual_label": ""
    })

manual_df = pd.DataFrame(manual_results)
manual_df


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

,id,bug_type,input_code,predicted_fix,manual_label
0,1,Syntax Error,def greet(name)\n print('Hello ' + name)\n\...,def greet(name):\n print('Hello ' + name)\n...,
1,2,Loop Syntax,for i in range(3)\n print(i),for i in range(3):\n print(i),
2,3,Function Definition,def square(x)\n return x * x\n\nprint(squar...,def square(x):\n return x * x\n\nprint(squa...,
3,4,Mutable Default Argument,"def add_item(item, my_list=[]):\n my_list.a...","def add_item(item, my_list=None):\n if my_l...",
4,5,Recursion/Base Case,def factorial(n):\n return n * factorial(n-...,def factorial(n):\n if n == 0:\n ret...,
5,6,Class Definition,"class Student\n def __init__(self, name):\n...","class Student:\n def __init__(self, name):\...",
6,7,Conditional Syntax,x = 10\nif x > 5\n print('Greater'),x = 10\nif x > 5:\n print('Greater'),
7,8,Indentation Error,"def add(a, b):\nreturn a + b\n\nprint(add(2, 3))","def add(a, b):\n return a + b\n\nprint(add(...",
8,9,String + Integer Concatenation,age = 20\nprint('Age is ' + age),age = 20\nprint('Age is ' + str(age)),
9,10,Object Method Syntax,"class Car:\n def __init__(self, brand):\n ...","class Car:\n def __init__(self, brand):\n ...",


In [ ]:
manual_df.to_csv("/content/manual_evaluation_sheet.csv", index=False)
print("Saved manual evaluation sheet to /content/manual_evaluation_sheet.csv")


Saved manual evaluation sheet to /content/manual_evaluation_sheet.csv


In [ ]:
manual_df.loc[manual_df["id"] == 1, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 2, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 3, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 4, "manual_label"] = "Partially Fixed"
manual_df.loc[manual_df["id"] == 5, "manual_label"] = "Failed"
manual_df.loc[manual_df["id"] == 6, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 7, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 8, "manual_label"] = "Fully Fixed"
manual_df.loc[manual_df["id"] == 9, "manual_label"] = "Partially Fixed"
manual_df.loc[manual_df["id"] == 10, "manual_label"] = "Fully Fixed"

manual_df


,id,bug_type,input_code,predicted_fix,manual_label
0,1,Syntax Error,def greet(name)\n print('Hello ' + name)\n\...,def greet(name):\n print('Hello ' + name)\n...,Fully Fixed
1,2,Loop Syntax,for i in range(3)\n print(i),for i in range(3):\n print(i),Fully Fixed
2,3,Function Definition,def square(x)\n return x * x\n\nprint(squar...,def square(x):\n return x * x\n\nprint(squa...,Fully Fixed
3,4,Mutable Default Argument,"def add_item(item, my_list=[]):\n my_list.a...","def add_item(item, my_list=None):\n if my_l...",Partially Fixed
4,5,Recursion/Base Case,def factorial(n):\n return n * factorial(n-...,def factorial(n):\n if n == 0:\n ret...,Failed
5,6,Class Definition,"class Student\n def __init__(self, name):\n...","class Student:\n def __init__(self, name):\...",Fully Fixed
6,7,Conditional Syntax,x = 10\nif x > 5\n print('Greater'),x = 10\nif x > 5:\n print('Greater'),Fully Fixed
7,8,Indentation Error,"def add(a, b):\nreturn a + b\n\nprint(add(2, 3))","def add(a, b):\n return a + b\n\nprint(add(...",Fully Fixed
8,9,String + Integer Concatenation,age = 20\nprint('Age is ' + age),age = 20\nprint('Age is ' + str(age)),Partially Fixed
9,10,Object Method Syntax,"class Car:\n def __init__(self, brand):\n ...","class Car:\n def __init__(self, brand):\n ...",Fully Fixed


In [ ]:
total_cases = len(manual_df)

fully_fixed_rate = (manual_df["manual_label"] == "Fully Fixed").mean() * 100
partial_fixed_rate = (manual_df["manual_label"] == "Partially Fixed").mean() * 100
failed_rate = (manual_df["manual_label"] == "Failed").mean() * 100
partial_or_better_rate = ((manual_df["manual_label"] == "Fully Fixed") | (manual_df["manual_label"] == "Partially Fixed")).mean() * 100

print(f"Total Manual Test Cases: {total_cases}")
print(f"Fully Fixed Rate: {fully_fixed_rate:.2f}%")
print(f"Partially Fixed Rate: {partial_fixed_rate:.2f}%")
print(f"Failed Rate: {failed_rate:.2f}%")
print(f"Partial-or-Better Rate: {partial_or_better_rate:.2f}%")


Total Manual Test Cases: 10
Fully Fixed Rate: 70.00%
Partially Fixed Rate: 20.00%
Failed Rate: 10.00%
Partial-or-Better Rate: 90.00%
